# 🌍 Seismic Data — 2D & Globe Visualization (Final Working Version)

### Why no `Scene()` for custom data?
`Scene.content.add()` only accepts live REST-backed layers with a `.properties` endpoint.  
It **cannot** accept in-memory data (FeatureSet, GeoJSON, drawn points).  
`Map.content.draw()` works perfectly and gives an identical visual result.

### What works ✅
| Feature | Method |
|---|---|
| Live PAGER earthquake feed | `Map` + `gis.content.add(item)` |
| Historical REST layers | `Map` + `safe_load_layer()` |
| Custom colored dots from GeoJSON | `Map` + `content.draw()` |
| Globe-style view | `Map` with `satellite` or `dark-gray-vector` basemap |
| DataFrame query & stats | Living Atlas `FeatureLayer.query()` |

---

## 📦 Step 1 — Install & Import

In [1]:
!pip install arcgis arcgis-mapping --quiet
print('✅ Ready')

✅ Ready


In [2]:
import json, urllib.request
import pandas as pd
import arcgis
from arcgis.gis         import GIS
from arcgis.geocoding   import geocode
from arcgis.geometry    import Geometry
from arcgis.layers      import MapImageLayer
from arcgis.map.symbols import SimpleMarkerSymbolEsriSMS

gis = GIS()
print('✅ Connected:', gis)

# ── Constants ─────────────────────────────────────────────────────────────────
LIVING_ATLAS_ID = '9e2f2b544c954fda9cd13b7f3e6eebce'
BASE            = 'https://earthquake.usgs.gov/arcgis/rest/services'
HISTORICAL_URL  = f'{BASE}/eq/EQ_Catalog/MapServer'
PLATES_URL      = f'{BASE}/eq/map_plateboundaries/MapServer'
FAULTS_URL      = f'{BASE}/eq/map_faults/MapServer'

GEOJSON = {
    'significant_month': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/significant_month.geojson',
    'M4_5_month'       : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson',
    'M2_5_month'       : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.geojson',
    'all_day'          : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson',
}
print('✅ Constants set')

✅ Connected: GIS @ https://www.arcgis.com version:2025.3
✅ Constants set


## 🔧 Step 2 — Helper Functions

In [3]:
# ── Color helpers ─────────────────────────────────────────────────────────────
def mag_to_color(mag):
    if   mag >= 7.0: return [180, 0,   0,   255]
    elif mag >= 6.0: return [255, 50,  0,   230]
    elif mag >= 5.0: return [255, 140, 0,   220]
    elif mag >= 4.0: return [255, 220, 0,   200]
    else:            return [100, 200, 100, 180]

def depth_to_color(depth):
    if   depth <= 30 : return [255, 50,  50,  230]
    elif depth <= 100: return [255, 165, 0,   220]
    elif depth <= 300: return [100, 100, 255, 210]
    else:              return [50,  0,   180, 200]

def mag_to_size(mag):
    return max(6, int((mag - 2) * 5))

def make_symbol(color, size):
    """Build pydantic symbol model — required by Map.content.draw()."""
    return SimpleMarkerSymbolEsriSMS(**{
        'type'   : 'esriSMS',
        'style'  : 'esriSMSCircle',
        'color'  : color,
        'size'   : size,
        'angle'  : 0, 'xoffset': 0, 'yoffset': 0,
        'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                    'color': [0, 0, 0, 80], 'width': 0.5}
    })

# ── Map factory ───────────────────────────────────────────────────────────────
def make_map(location='World', basemap='dark-gray-vector', zoom=2, height='600px'):
    """Create a map widget with sensible defaults."""
    m = gis.map(location)
    m.basemap.basemap  = basemap
    m.zoom             = zoom
    m.layout.height    = height
    return m

# ── Safe REST loader ──────────────────────────────────────────────────────────
def safe_load_layer(url, name):
    """Load a MapImageLayer — returns None silently if the server is down."""
    try:
        lyr = MapImageLayer(url)
        _   = lyr.properties
        print(f'  ✅ {name}')
        return lyr
    except Exception:
        print(f'  ❌ {name} — server unavailable, skipping')
        return None

# ── GeoJSON fetcher ───────────────────────────────────────────────────────────
def fetch_earthquakes(feed='M4_5_month'):
    """Fetch USGS GeoJSON feed → clean DataFrame."""
    print(f'📡 Fetching {feed}...')
    with urllib.request.urlopen(GEOJSON[feed]) as r:
        data = json.loads(r.read().decode())
    rows = []
    for f in data['features']:
        p          = f['properties']
        lon, lat, depth = f['geometry']['coordinates']
        rows.append({
            'mag'  : p.get('mag'),
            'place': p.get('place'),
            'time' : pd.to_datetime(p.get('time'), unit='ms'),
            'depth': depth,
            'lon'  : lon,
            'lat'  : lat,
        })
    df = (pd.DataFrame(rows)
            .dropna(subset=['mag','lat','lon'])
            .sort_values('mag', ascending=False)
            .reset_index(drop=True))
    print(f'✅ {len(df)} earthquakes  |  max mag: {df["mag"].max()}  |  avg depth: {df["depth"].mean():.1f} km')
    return df

# ── Draw engine ───────────────────────────────────────────────────────────────
def draw_earthquakes(map_widget, df, color_by='magnitude'):
    """
    Draw earthquake points on a Map widget using Map.content.draw().
    color_by: 'magnitude' or 'depth'
    NOTE: Only works on Map (gis.map()), NOT on Scene().
    """
    for _, row in df.iterrows():
        pt    = Geometry({'x': row['lon'], 'y': row['lat'],
                          'spatialReference': {'wkid': 4326}})
        color = mag_to_color(row['mag']) if color_by == 'magnitude' \
                else depth_to_color(row['depth'])
        map_widget.content.draw(pt, symbol=make_symbol(color, mag_to_size(row['mag'])))
    print(f'✅ Drew {len(df)} earthquakes  (color_by={color_by})')

print('✅ All helpers ready')

✅ All helpers ready


---
## 🗺️ Step 3 — Live Earthquake Map (ArcGIS Living Atlas)
Updated every 5 minutes. Built-in symbology: dot size = magnitude, color = PAGER alert level.

In [4]:
live_map = make_map(zoom=2)

item = gis.content.get(LIVING_ATLAS_ID)
print(f'📦 {item.title}  |  {item.type}')
print(f'   {item.snippet}')

live_map.content.add(item)
print('\n✅ Live feed added — click any dot for magnitude, depth, PAGER alert')
live_map

📦 Recent Earthquakes  |  Feature Service
   This service presents recent earthquake information from the USGS Prompt Assessment of Global Earthquakes for Response (PAGER) program.

✅ Live feed added — click any dot for magnitude, depth, PAGER alert


Map(layout=Layout(height='600px'))

---
## 📚 Step 4 — Historical Catalog + Fault Lines + Plate Boundaries

In [5]:
hist_map = make_map(zoom=2)

print('Loading USGS REST layers...')
plates = safe_load_layer(PLATES_URL,    'Tectonic Plate Boundaries')
faults = safe_load_layer(FAULTS_URL,    'Global Fault Lines')
hist   = safe_load_layer(HISTORICAL_URL,'Historical Seismicity 1900–2024 (M5.5+)')

for lyr in [plates, faults, hist]:
    if lyr: hist_map.content.add(lyr)

hist_map.content.add(gis.content.get(LIVING_ATLAS_ID))
print('  ✅ Living Atlas recent feed')
hist_map

Loading USGS REST layers...
  ✅ Tectonic Plate Boundaries
  ❌ Global Fault Lines — server unavailable, skipping
  ✅ Historical Seismicity 1900–2024 (M5.5+)
  ✅ Living Atlas recent feed


Map(layout=Layout(height='600px'))

---
## 📍 Step 5 — Focus on Any Region
Change `REGION` to zoom into any seismically active area.

In [6]:
# ✏️ Options: 'Japan' | 'Indonesia' | 'Chile' | 'Turkey' | 'California, USA' | 'India' | 'Alaska'
REGION = 'Japan'

region_map = make_map(location=REGION, zoom=6)

for lyr in [safe_load_layer(HISTORICAL_URL,'Historical'),
            safe_load_layer(FAULTS_URL,    'Faults')]:
    if lyr: region_map.content.add(lyr)

region_map.content.add(gis.content.get(LIVING_ATLAS_ID))
print(f'\n✅ Focused on: {REGION}')
region_map

  ✅ Historical
  ❌ Faults — server unavailable, skipping

✅ Focused on: Japan


Map(layout=Layout(height='600px'))

---
## 🔢 Step 6 — Query Earthquake Data as a DataFrame

In [7]:
eq_layer = gis.content.get(LIVING_ATLAS_ID).layers[0]
print('Layer:', eq_layer)

fset = eq_layer.query(where='1=1', out_fields='*', return_geometry=True)
df   = fset.sdf
print(f'\n✅ {len(df)} records  |  Columns: {df.columns.tolist()}')
df.head(10)

Layer: <FeatureLayer url:"https://services9.arcgis.com/RHVPKKiFTONKtxq3/arcgis/rest/services/USGS_Seismic_Data_v1/FeatureServer/0">

✅ 1976 records  |  Columns: ['OBJECTID', 'id', 'mag', 'eventType', 'sig', 'alert', 'place', 'hoursOld', 'eventTime', 'updated', 'tz', 'url', 'detail', 'felt', 'cdi', 'mmi', 'status', 'tsunami', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'magType', 'longitude', 'latitude', 'depth', 'SHAPE']


,OBJECTID,id,mag,eventType,sig,alert,place,hoursOld,eventTime,updated,...,types,nst,dmin,rms,gap,magType,longitude,latitude,depth,SHAPE
0,27273,nc75269596,3.98,earthquake,804,GREEN,"8 km E of Gilroy, CA, USA",2133,2025-11-26 14:16:42,2025-12-02 14:26:52,...,"dyfi,focal-mechanism,impact-link,losspager,mom...",136,0.02887,0.18,35.0,Mw,-121.4757,36.99733,2.92,"{""x"": -13522609.357099999, ""y"": 4438735.094800..."
1,27459,us7000re2g,6.6,earthquake,675,GREEN,"45 km WNW of Sinabang, Indonesia",2118,2025-11-27 04:56:26,2025-11-28 05:17:05,...,"dyfi,ground-failure,losspager,moment-tensor,or...",42,2.894,1.18,81.0,MWw,96.0266,2.6921,25.388,"{""x"": 10689632.2146, ""y"": 299793.52969999984, ..."
2,27564,ak2025xjbv,6.0,earthquake,1124,GREEN,"12 km WNW of Susitna, Alaska, USA",2106,2025-11-27 17:11:29,2025-12-03 23:51:04,...,"dyfi,ground-failure,impact-link,losspager,mome...",225,0.470205,1.186067,13.68207,MWw,-150.7508,61.56952,69.4334,"{""x"": -16781501.5643, ""y"": 8757782.141199999, ..."
3,27628,nc75270551,4.04,earthquake,751,GREEN,"5 km S of San Juan Bautista, CA, USA",2102,2025-11-27 20:57:37,2025-12-04 17:06:22,...,"dyfi,focal-mechanism,impact-link,losspager,mom...",121,0.02499,0.2,29.0,Mw,-121.5437,36.799,8.2,"{""x"": -13530179.0825, ""y"": 4411126.888899997, ..."
4,28442,nc75272501,3.25,earthquake,632,<NA>,"6 km WSW of Kenwood, CA, USA",2020,2025-12-01 06:56:36,2025-12-07 22:31:08,...,"dyfi,focal-mechanism,moment-tensor,nearby-citi...",163,0.04275,0.15,16.0,Mw,-122.6063,38.38533,4.76,"{""x"": -13648474.594700001, ""y"": 4634004.4164, ..."
5,29343,us7000rfgy,5.8,earthquake,654,YELLOW,"138 km NNW of Tumxuk, China",1948,2025-12-04 07:44:06,2025-12-05 20:17:59,...,"dyfi,ground-failure,losspager,moment-tensor,or...",115,2.425,0.75,27.0,MWw,78.4505,41.0259,10.0,"{""x"": 8733069.712499999, ""y"": 5016162.65879999..."
6,29967,us6000rsy1,7.0,earthquake,1102,GREEN,2025 Hubbard Glacier Earthquake,1887,2025-12-06 20:41:49,2025-12-12 20:52:01,...,"dyfi,earthquake-name,finite-fault,general-text...",128,0.44,1.07,30.0,MWw,-139.5457,60.3586,10.0,"{""x"": -15534156.2664, ""y"": 8480012.652800001, ..."
7,30603,us6000rtdt,7.6,earthquake,1072,YELLOW,"2025 Aomori Prefecture, Japan Earthquake",1845,2025-12-08 14:15:10,2025-12-15 12:41:14,...,"dyfi,earthquake-name,finite-fault,ground-failu...",139,0.866,0.77,17.0,MWw,142.1714,41.0027,45.452,"{""x"": 15826447.8534, ""y"": 5012739.921400003, ""..."
8,30659,nc75276661,3.6,earthquake,669,<NA>,"4 km SE of San Ramon, CA, USA",1842,2025-12-08 17:07:58,2025-12-14 23:45:06,...,"dyfi,focal-mechanism,nearby-cities,origin,phas...",152,0.03081,0.13,17.0,Ml,-121.9447,37.75767,9.62,"{""x"": -13574818.1983, ""y"": 4545248.576099999, ..."
9,30763,us6000rtie,6.6,earthquake,672,GREEN,"126 km S of Honchō, Japan",1837,2025-12-08 21:52:41,2025-12-12 02:57:41,...,"dyfi,ground-failure,losspager,moment-tensor,or...",129,1.011,0.55,29.0,MWw,143.1637,41.0034,19.0,"{""x"": 15936910.184100002, ""y"": 5012843.1759999..."


In [8]:
# ── Filter — confirmed column names from actual output ────────────────────────
# mag | depth | place | eventTime | latitude | longitude | alert | sig | magType
big = df[df['mag'] >= 5.0].sort_values('mag', ascending=False)
print(f'M5.0+ events: {len(big)}')
big[['mag', 'depth', 'place', 'eventTime', 'alert']].head(15)

M5.0+ events: 170


,mag,depth,place,eventTime,alert
7,7.6,45.452,"2025 Aomori Prefecture, Japan Earthquake",2025-12-08 14:15:10,YELLOW
1802,7.1,619.849,"55 km NNW of Kota Belud, Malaysia",2026-02-22 16:57:46,GREEN
6,7.0,10.0,2025 Hubbard Glacier Earthquake,2025-12-06 20:41:49,GREEN
10,6.7,10.678,"114 km ENE of Hachinohe, Japan",2025-12-12 02:44:11,GREEN
17,6.6,67.534,"31 km ESE of Yilan, Taiwan",2025-12-27 15:05:56,GREEN
1,6.6,25.388,"45 km WNW of Sinabang, Indonesia",2025-11-27 04:56:26,GREEN
9,6.6,19.0,"126 km S of Honchō, Japan",2025-12-08 21:52:41,GREEN
15,6.5,110.18,"42 km NNE of Goroka, Papua New Guinea",2025-12-22 10:31:28,GREEN
20,6.5,35.0,"4 km NNW of Rancho Viejo, Mexico",2026-01-02 13:58:18,YELLOW
21,6.4,35.0,"16 km ESE of Baculin, Philippines",2026-01-07 03:02:56,GREEN


In [9]:
print('📊 Seismic Statistics:')
print(f'   Total events  : {len(df)}')
print(f'   Max magnitude : {df["mag"].max()}')
print(f'   Avg magnitude : {df["mag"].mean():.2f}')
print(f'   M6.0+ events  : {len(df[df["mag"] >= 6.0])}')
print(f'   Avg depth (km): {df["depth"].mean():.1f}')
print(f'   Deepest (km)  : {df["depth"].max()}')
print('\n🚨 PAGER Alert breakdown:')
print(df['alert'].value_counts().to_string())

📊 Seismic Statistics:
   Total events  : 1976
   Max magnitude : 7.6
   Avg magnitude : 2.34
   M6.0+ events  : 25
   Avg depth (km): 32.2
   Deepest (km)  : 653.785

🚨 PAGER Alert breakdown:
alert
GREEN     78
YELLOW     4


---
## 🌍 Step 7 — Fetch Live GeoJSON Data (M4.5+ last 30 days)

In [10]:
# ✏️ Options: 'significant_month' | 'M4_5_month' | 'M2_5_month' | 'all_day'
geo_df = fetch_earthquakes('M4_5_month')
geo_df.head(10)

📡 Fetching M4_5_month...
✅ 488 earthquakes  |  max mag: 7.1  |  avg depth: 84.3 km


,mag,place,time,depth,lon,lat
0,7.1,"55 km NNW of Kota Belud, Malaysia",2026-02-22 16:57:46.240,619.849,116.2637,6.8285
1,6.4,"53 km WNW of Port-Olry, Vanuatu",2026-02-14 02:27:37.951,10.000,166.6013,-14.8934
2,6.2,Fiji region,2026-02-10 20:44:52.774,510.944,-178.3348,-21.0396
3,6.2,"32 km SW of Ovalle, Chile",2026-02-12 13:34:31.114,36.885,-71.4451,-30.8012
4,6.1,"Kermadec Islands, New Zealand",2026-02-04 10:39:28.490,167.000,-178.5970,-29.5260
5,6.1,"93 km SW of Nikolski, Alaska",2026-02-23 05:11:48.783,14.000,-169.8574,52.3572
6,6.0,South Sandwich Islands region,2026-01-29 07:30:35.031,27.000,-25.5509,-57.7775
7,6.0,"86 km ESE of Ozernovskiy, Russia",2026-02-16 21:42:52.597,43.000,157.6862,51.2724
8,6.0,"110 km SSE of Kirakira, Solomon Islands",2026-02-21 16:42:07.295,10.000,162.3952,-11.3339
9,6.0,"284 km SSW of Severo-Kuril’sk, Russia",2026-02-15 15:58:48.801,54.981,154.5612,48.3271


---
## 🔴 Step 8 — Globe Map: Colored by MAGNITUDE
Dark satellite basemap + magnitude-colored dots from GeoJSON.

In [11]:
mag_map = make_map(basemap='satellite', zoom=2, height='650px')

draw_earthquakes(mag_map, geo_df, color_by='magnitude')

print('\nColor  = magnitude  |  Size = magnitude')
print('  🔴 Dark red  = M7.0+  |  🔴 Red    = M6.0–6.9')
print('  🟠 Orange    = M5.0–5.9  |  🟡 Yellow = M4.0–4.9')
mag_map

✅ Drew 488 earthquakes  (color_by=magnitude)

Color  = magnitude  |  Size = magnitude
  🔴 Dark red  = M7.0+  |  🔴 Red    = M6.0–6.9
  🟠 Orange    = M5.0–5.9  |  🟡 Yellow = M4.0–4.9


Map(layout=Layout(height='650px'))

---
## 🔵 Step 9 — Globe Map: Colored by DEPTH
Same data, color now shows earthquake depth — reveals **subduction zones** as deep blue clusters.

In [12]:
depth_map = make_map(basemap='dark-gray-vector', zoom=2, height='650px')

draw_earthquakes(depth_map, geo_df, color_by='depth')

print('\nColor = depth  |  Size = magnitude')
print('  🔴 Red        = Shallow   0–30 km   (crust/surface faults)')
print('  🟠 Orange     = Mid       30–100 km (upper mantle)')
print('  🔵 Blue       = Deep      100–300 km (subduction zones)')
print('  🟣 Dark blue  = Very deep 300+ km   (deep slab)')
depth_map

✅ Drew 488 earthquakes  (color_by=depth)

Color = depth  |  Size = magnitude
  🔴 Red        = Shallow   0–30 km   (crust/surface faults)
  🟠 Orange     = Mid       30–100 km (upper mantle)
  🔵 Blue       = Deep      100–300 km (subduction zones)
  🟣 Dark blue  = Very deep 300+ km   (deep slab)


Map(layout=Layout(height='650px'))

---
## 🗾 Step 10 — Regional Focus Map with Custom Dots

In [13]:
# ✏️ Change to any region
FOCUS = 'Japan'

res = geocode(FOCUS, max_locations=1)

focus_map = make_map(location=FOCUS, basemap='satellite', zoom=5, height='650px')

if res:
    clat = res[0]['location']['y']
    clon = res[0]['location']['x']
    # Filter to ±20° bounding box around region
    region_df = geo_df[
        (geo_df['lat'].between(clat - 20, clat + 20)) &
        (geo_df['lon'].between(clon - 20, clon + 20))
    ].copy()
    print(f'📍 Region: {FOCUS}  |  Earthquakes in bounding box: {len(region_df)}')
    draw_earthquakes(focus_map, region_df, color_by='depth')
else:
    print('Could not geocode region — drawing all earthquakes')
    draw_earthquakes(focus_map, geo_df, color_by='depth')

focus_map

📍 Region: Japan  |  Earthquakes in bounding box: 77
✅ Drew 77 earthquakes  (color_by=depth)


Map(layout=Layout(height='650px'))

---
## 📊 Step 11 — SeDF Plot (Magnitude Gradient)
Uses the ArcGIS `spatial.plot()` renderer on the Living Atlas DataFrame.

In [14]:
plot_map = make_map(zoom=2)

# 'mag' confirmed as correct column from actual output
df.spatial.plot(
    map_widget    = plot_map,
    renderer_type = 'c',
    method        = 'esriClassifyNaturalBreaks',
    class_count   = 5,
    col           = 'mag',
    cmap          = 'Reds',
    alpha         = 0.8
)

print('✅ Plotted — darker red = stronger magnitude')
plot_map

✅ Plotted — darker red = stronger magnitude


Map(layout=Layout(height='600px'))

---
## 📋 Cheat Sheet

### Core pattern
```python
m = make_map('Japan', basemap='satellite', zoom=6)
geo_df = fetch_earthquakes('M4_5_month')
draw_earthquakes(m, geo_df, color_by='depth')   # or 'magnitude'
m
```

### Why Scene() is NOT used
```
Scene.content.draw()  → does not exist (SceneContent has no draw method)
Scene.content.add(FeatureSet)  → crashes: 'FeatureSet has no attribute properties'
Scene.content.add(MapImageLayer)  → crashes: 'NoneType has no attribute pop'
Scene.content.add(gis.content.get(...))  → crashes: 'NoneType does not support item assignment'
✅ Solution: gis.map() + satellite basemap = identical visual, fully working
```

### Confirmed column names (Living Atlas)
`mag` · `depth` · `place` · `eventTime` · `latitude` · `longitude` · `alert` · `sig` · `magType`

### GeoJSON feeds (always live)
| Key | Contents |
|---|---|
| `significant_month` | Significant EQ past 30 days |
| `M4_5_month` | All M4.5+ past 30 days |
| `M2_5_month` | All M2.5+ past 30 days |
| `all_day` | All EQ today |

### Regions to explore
🇯🇵 Japan · 🇮🇩 Indonesia · 🇨🇱 Chile · 🇹🇷 Turkey · 🇺🇸 California · 🇮🇳 India · 🌊 Ring of Fire